<a href="https://colab.research.google.com/github/Kubojah-Dan/kuboja-codeboosters-2026/blob/main/DAY6/GenAI_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Generative AI** - *Prompt Engineering*

In [1]:
#============================
# Install Libraries
#============================

!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Libraties Ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00
Libraties Ready


In [3]:
#==========================
# Configure Groq API
#==========================

from groq import Groq

API_KEY = "****************************"
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"

print(f"Groq Client configured with model: {MODEL}")

Groq Client configured with model: llama-3.1-8b-instant


In [5]:
#===========================
# LLM API Call
#===========================

def ask_llm(user_message, system_message="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
  """
     Send a message to the LLM and return the response text.

 """
  response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        stream=False
    )
  return response.choices[0].message.content

test_response = ask_llm(
    "."
)
print("=== LLM Response ===")
print(test_response)


=== LLM Response ===
There are several individuals and concepts known as "Daniel." Here are a few possibilities:

1. **Daniel (biblical figure)**: Daniel is a major figure in the Hebrew Bible and the Christian Old Testament. He was a young Jewish noble who was taken captive by the Babylonians and was known for his wisdom and faith in God. Daniel is famous for his interpretations of dreams and his ability to survive the lions' den.
2. **Daniel (name)**: Daniel is a popular given name that has been used in various cultures for centuries. It is derived from the Hebrew name "Daniel," which means "God is my judge."
3. **Daniel (biblical book)**: The Book of Daniel is a biblical book that consists of 12 chapters. It is a collection of apocalyptic visions and stories that were written to encourage Jews in the face of persecution.
4. **Daniel (historical figure)**: There are several historical figures known as Daniel, including Daniel Boone, an American frontiersman and explorer, and Daniel We

In [6]:
#====================================
# Understand tokens and context
#====================================

response_etl = ask_llm(
    "In 3 bullet points, explain how the medallion Architecture"
    "(Bronze, Silver, Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineering instructor."
                  "Be concise and practical."
)
print("Medallion + ETL Architecture")
print(response_etl)
print()
print("The model above used approximately", len(response_etl.split())*1.3, "tokens")

Medallion + ETL Architecture
Here are 3 key points explaining the Medallion Architecture's relation to ETL (Extract, Transform, Load) pipelines:

* **Bronze Layer: Data Ingestion**: In the Medallion Architecture, the Bronze layer represents the initial data ingestion point, where raw data is extracted from various sources (e.g., databases, APIs, files). This step is equivalent to the ETL pipeline's Extract phase, where data is fetched from the source systems.
* **Silver Layer: Data Processing and Transformation**: The Silver layer is where data is processed, transformed, and enriched, making it ready for analysis. This step aligns with the ETL pipeline's Transform phase, where data is cleaned, aggregated, and formatted according to the target schema or business requirements.
* **Gold Layer: Data Warehousing and Analytics**: The Gold layer serves as the centralized data repository, storing the transformed data from the Silver layer. This step corresponds to the ETL pipeline's Load phase

In [8]:
response_etl = ask_llm(
    "I would like to work on a project on AI meal Planner"
    "I would like you to give me the recommendation of features to have in my project",
    system_message="You are a senior Full Stack developer."
                   "Answer precisely and accurately"

)
print("Medallion + ETL Architecture")
print(response_etl)
print()
print("The model above used approximately", len(response_etl.split())*1.3, "tokens")

Medallion + ETL Architecture
An AI meal planner project sounds like a great idea. Here are some features that you may want to consider including:

**Core Features**

1. **User Profile**: Allow users to create a profile, including their dietary preferences, restrictions, and allergies.
2. **Meal Planning**: Enable users to input their meal requirements (e.g., number of people, meals per day) and receive a personalized meal plan.
3. **Recipe Database**: Develop a database of recipes with nutritional information, cooking instructions, and images.
4. **Meal Suggestions**: Use natural language processing (NLP) and machine learning algorithms to generate meal suggestions based on user preferences and dietary needs.

**Advanced Features**

1. **Grocery List Generation**: Automatically generate a grocery list based on the meal plan and user preferences.
2. **Nutrition Analysis**: Provide detailed nutrition information, including calorie counts, macronutrients, and allergens.
3. **Cooking Instr

In [17]:
#===================================
# Experiment 1: Zero-Shot Prompting
#===================================

zero_shot_response = ask_llm(
    "Extract the city name from this address: "
    "456 Brigrade Road, Bangalore 500025, Karnataka, India"
)
print("Zero-Shot Response:")
print(zero_shot_response)
print()


ambigous_response = ask_llm("Clean this data: ramesh kumar, 45000, mumbai")
print("Ambigous Response:")
print(ambigous_response)

Zero-Shot Response:
The city name is "Bangalore".

Ambigous Response:
I can help clean the data. Based on the given data "ramesh kumar, 45000, mumbai", it appears to be a list of a person's name, salary, and location. 

Here's the cleaned data:

- Name: Ramesh Kumar
- Salary: 45,000
- Location: Mumbai

Note that I've capitalized the first letter of the name as per conventional naming practices, and separated the salary with a comma to improve readability.


In [16]:
#================================
# Experiment 2: Few-Shot Prompt
#================================

few_shot_json = """ Extract city name from address
  {
        {"address": "456 Brigade Road, Bangalore 500025, Karnataka, India", "city": "Bangalore"},
        {"address": "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA", "city": "Mountain View"},
        {"address": "Eiffel Tower, Champ de Mars, 75007 Paris, France", "city": "Paris"}
}
"""
few_shot_response = ask_llm(few_shot_json, system_message="Respond only with the city name.", temperature=0.0)
print("Few-Shot Response:")
print(few_shot_response)

Few-Shot Response:
Bangalore
Mountain View
Paris


In [18]:
#====================================
# Experiment 3: Role Prompting
#====================================

same_question = "Review this python code and identify any issues:\n " \
                "df['revenue'] = df['qty'] * df['price']\n"\
                "print(df)"

# Without Role
generic_response = ask_llm(same_question, temperature=0.2)
print("Without Role Prompting:")
print(generic_response[:300], '....')
print()

# With Role
role_response =ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production.",
    temperature=0.2
)
print("With Role Prompting:")
print(role_response[:300], '....')

Without Role Prompting:
The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are a few potential issues that could arise:

1. **Missing Import Statement**: The code uses the pandas library, but it does not include the import statement. To fix this, add `import pand ....

With Role Prompting:
**Code Review**

The provided Python code appears to be a simple data manipulation task. However, there are a few potential issues to consider:

### 1. Missing Error Handling

The code does not handle potential errors that may occur during the execution. For example, if the 'qty' or 'price' columns  ....


In [20]:
#===============================
# Temperature Experiment
#===============================

prompt = "Give me one creative name for a data analytics startup"
print("=== Temperature Experiment ===")
for temp in [0.0, 0.5, 0.1]:
  response = ask_llm(prompt, temperature=temp)
  print(f"Temperature: {temp}\nResponse: {response}\n")
  time.sleep(1)

print()
print("Observation:")
print(" temperature=0.0 > same or very similar answer every run (deterministic)")
print(" temperature=0.5 > some variation")
print(" temperature=1.0 > more creative/varied, sometimes surprising")

=== Temperature Experiment ===
Temperature: 0.0
Response: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.

Temperature: 0.5
Response: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" comes from the word "nexus," meaning a connection or link between things, which represents the connection between data and insights. It also has a modern and sleek sound to it. "Insights" clearly communicates the focus of the startup on providing valuable and actionable information to its clients.

Alternatively, you may also consider other options like:

- **PulseData**
- **Apex Analytics**
- **Cerebro Insights** (Cerebro is Spanish for "brain")
- **Kairos Analytics** (Kairos is Greek for "oppo

In [21]:
messy_invoices = [
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop Purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for Office Cleaning Services",
    "INV-2024-103 | arjun consultancy | 8000 | march 15 2024 | python training",
    "SURETH RAD HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20"
]
print("=== Messy Invoices ===")
for i, inv in enumerate(messy_invoices, 1):
  print(f"Invoice {i}: {inv}")
print(f"\nTotal: {len(messy_invoices)}")

=== Messy Invoices ===
Invoice 1: INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop Purchase
Invoice 2: Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for Office Cleaning Services
Invoice 3: INV-2024-103 | arjun consultancy | 8000 | march 15 2024 | python training
Invoice 4: SURETH RAD HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20

Total: 4


In [22]:
#======================================
# Process all Invoices with LLM
#======================================

def extract_invoice_data(invoice_text, system_prompt, client, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Extract data from: {invoice_text}"}
        ],
        temperature=0.0,
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

system_prompt = """
You are an expert data extractor. Extract the following fields from the invoice text into a valid JSON object:
- invoice_id
- vendor_name
- date
- amount
- description
"""

print("=== Processing Invoices ===\n")
for inv in messy_invoices:
    try:
        result = extract_invoice_data(inv, system_prompt, client, MODEL)
        print(f"Raw: {inv}")
        print(f"Extracted: {result}\n")
        time.sleep(1)
    except Exception as e:
        print(f"Error processing {inv}: {e}")

=== Processing Invoices ===

Raw: INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop Purchase
Extracted: {
  "invoice_id": "INV-2024-0891",
   "vendor_name": "TECHWORLD SOLUTIONS",
   "date": "15th Jan 2024",
   "amount": 45000,
   "description": "Laptop Purchase"
}

Raw: Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for Office Cleaning Services
Extracted: {
  "invoice_id": null,
   "vendor_name": "PRIYA ENTERPRISES",
   "date": "07-02-2024",
   "amount": 12500,
   "description": "Office Cleaning Services"
}

Raw: INV-2024-103 | arjun consultancy | 8000 | march 15 2024 | python training
Extracted: {
  "invoice_id": "INV-2024-103",
   "vendor_name": "arjun consultancy",
   "date": "march 15 2024",
   "amount": 8000,
   "description": "python training"
}

Raw: SURETH RAD HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
Extracted: {
  "invoice_id": null,
   "vendor_name": "SURETH RAD HARDWARE STORE",
   "date": "2024/01/20",
   "amount": 25000,
   "de

In [26]:
#=====================================================
# Process all invoices with LLM (Vendor, Amount, Date)
#=====================================================

def extract_invoice_data(invoice_text, system_prompt, client, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Extract data from: {invoice_text}"}
        ],
        temperature=0.0,
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

system_prompt_limited = """
You are a data extraction tool. Extract only these fields into a JSON object:
- vendor
- amount
- date
"""

print("=== Processing All Invoices (Limited Fields) ===\n")
for inv in messy_invoices:
    try:
        result = extract_invoice_data(inv, system_prompt_limited, client, MODEL)
        print(f"Raw: {inv}")
        print(f"Extracted: {result}\n")
        time.sleep(1)
    except Exception as e:
        print(f"Error processing: {e}")

=== Processing All Invoices (Limited Fields) ===

Raw: INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop Purchase
Extracted: {
  "vendor": "TECHWORLD SOLUTIONS",
   "amount": 45000,
   "date": "15th Jan 2024"
}

Raw: Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for Office Cleaning Services
Extracted: {
  "vendor": "PRIYA ENTERPRISES",
   "amount": 12500,
   "date": "2024-02-07"
}

Raw: INV-2024-103 | arjun consultancy | 8000 | march 15 2024 | python training
Extracted: {
  "vendor": "arjun consultancy",
   "amount": 8000,
   "date": "march 15 2024"
}

Raw: SURETH RAD HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
Extracted: {
  "vendor": "SURETH RAD HARDWARE STORE",
   "amount": 25000,
   "date": "2024/01/20"
}



In [34]:
#=====================================================================
# Few-Shot Prompt that tracks multiple names and salaries in JSON format
#=====================================================================

few_shot_hr_prompt = """
Your task is to extract ALL names and salaries from the provided text.
Respond with a JSON object containing a list called 'employees'.

Examples:
Text: "The team lead, Rahul Sharma, earns 1200000, while the intern Amit earns 20000."
JSON: {"employees": [{"name": "Rahul Sharma", "salary": 1200000}, {"name": "Amit", "salary": 20000}]}
---
Text: "We hired Sarah Jenkins (85000) and ramesh kumar (45000)."
JSON: {"employees": [{"name": "Sarah Jenkins", "salary": 85000}, {"name": "Ramesh Kumar", "salary": 45000}]}
---
Text: "The developer Anita Roy is drawing 1500000 yearly. Also, John Doe has a salary of 95k."
JSON:
"""

hr_response = ask_llm(
    few_shot_hr_prompt,
    system_message="You are a data extraction assistant. Always respond with a JSON array of objects inside an 'employees' key.",
    temperature=0.3
)

print("Extracted HR Data:")
print(hr_response)

Extracted HR Data:
{
  "employees": [
    {"name": "Anita Roy", "salary": 1500000},
    {"name": "John Doe", "salary": 95000}
  ]
}
